# Hybrid Search (BM25 + Dense): two lenses, fused — built from scratch

Dense embeddings match on **meaning** but lose the razor edge on **exact tokens** — product codes, error codes, rare names. A lexical retriever (**BM25**) nails exact terms but is blind to **paraphrase**. Each lens has a real, *complementary* blind spot. **Hybrid search** runs both and **fuses** their results so a passage strong on *either* lens still surfaces.

This notebook builds it from primitives on the chapter corpus (chapter 1's passages + a few blind-spot passages):
- **BM25** from scratch (the Robertson–Zaragoza / Lucene scoring function) — the lexical lens;
- a **dense** bi-encoder (all-MiniLM-L6-v2, chapter 3's production embedder) — the semantic lens;
- two **fusion** families: weighted-sum (min-max normalized convex combination, `alpha`) and **Reciprocal Rank Fusion (RRF)**.

Every number the concept page quotes is printed by a cell below. We assert before we claim.

## Step 1 — set up: import the canonical hybrid-search functions

Everything below uses the verified functions from `hybrid_search.py` — the same code the concept page shows — so there's no drift between page, script, and notebook. The dense lens loads `all-MiniLM-L6-v2` on CPU (deterministic); if it's unavailable the module falls back to a lexical embedder and says so.

In [1]:
import numpy as np

from hybrid_search import (
    BM25_K1, BM25_B, RRF_K, NEUTRAL_ALPHA, TUNED_ALPHA, TOP_K,
    BM25, DenseRetriever, full_corpus, build_probes,
    weighted_sum_fusion, reciprocal_rank_fusion, min_max_normalize,
    reciprocal_rank, recall_at_k, evaluate, alpha_sweep,
)

print('numpy:', np.__version__)
try:
    import torch
    device = ('cuda' if torch.cuda.is_available()
              else 'mps' if torch.backends.mps.is_available() else 'cpu')
    print('torch:', torch.__version__, '| device:', device)
except ImportError:
    print('torch: not installed')

corpus = full_corpus()
bm25 = BM25(corpus)
dense = DenseRetriever(corpus)
probes = build_probes(corpus)
print('corpus passages:', len(corpus), '| dense lens:', dense.backend)
print(f'BM25 k1={BM25_K1}, b={BM25_B}, avgdl={bm25.avgdl:.2f} | RRF k={RRF_K}')

numpy: 2.4.6


torch: 2.12.0 | device: mps


corpus passages: 11 | dense lens: sentence-transformers/all-MiniLM-L6-v2
BM25 k1=1.2, b=0.75, avgdl=12.55 | RRF k=60


## Step 2 — the corpus and the two blind-spot probes

The corpus is chapter 1's eight passages plus three we add on purpose:
- `doc[8]` — a **terse exact-code** line (`Error E-4011 …`);
- `doc[9]` — a **paraphrase** line (`Climbing steadily, … rose skyward …`) a paraphrased query never repeats word-for-word;
- `doc[10]` — a **chatty same-topic distractor** (lots of "errors / warnings", but no code).

Two probes each defeat one lens:
- **exact-code probe** — the terse `doc[8]` is what we want, but a chatty distractor is more *topically central*, so a dense model can rank the distractor first;
- **paraphrase probe** — its gold `doc[9]` shares **no content word** with the query, so BM25 scores it exactly 0.

In [2]:
for i, d in enumerate(corpus):
    tag = ''
    if i == 8: tag = '  <- exact-code gold'
    elif i == 9: tag = '  <- paraphrase gold'
    elif i == 10: tag = '  <- chatty distractor'
    print(f'doc[{i:>2}] {d}{tag}')

print()
for p in probes:
    print(f'PROBE [{p.label}]  gold=doc[{p.gold}]')
    print(f'   {p.query}')

doc[ 0] The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.
doc[ 1] Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
doc[ 2] The project lead for Helios-7 is Dr. Amara Okoye, based in the Nairobi office.
doc[ 3] Helios-7 completes one orbit of Earth every 97 minutes in a sun-synchronous orbit.
doc[ 4] Photosynthesis converts carbon dioxide and water into glucose using sunlight.
doc[ 5] The Eiffel Tower in Paris was completed in 1889 for the World's Fair.
doc[ 6] A standard chessboard has 64 squares arranged in an 8 by 8 grid.
doc[ 7] Water boils at 100 degrees Celsius at standard atmospheric pressure.
doc[ 8] Error E-4011 appeared in the Helios-7 telemetry stream.  <- exact-code gold
doc[ 9] Climbing steadily, Helios-7 rose skyward moments past liftoff.  <- paraphrase gold
doc[10] The Helios-7 ground team spent the afternoon investigating several telemetry errors and console warnings.  <- chatty distractor

PROBE [exact-code (d

## Step 3 — BM25 from scratch: IDF, saturation, length-norm

BM25 scores a document for a query as a sum over query terms:

$$\text{BM25}(q,d)=\sum_{t\in q}\text{IDF}(t)\cdot\frac{tf_{t,d}\,(k_1+1)}{tf_{t,d}+k_1\,(1-b+b\,\frac{|d|}{\text{avgdl}})}$$

`tf` is the term's count in the doc, `|d|` the doc length, `avgdl` the average length. `k1` controls **saturation** (the 11th occurrence adds far less than the 2nd), `b` controls **length normalization**. We print a passage's score and the IDF of a rare vs a common token.

In [3]:
# IDF: rare tokens weigh far more than common ones
for tok in ['e', 'helios', 'the', 'telemetry']:
    print(f'IDF({tok!r}) = {bm25.idf.get(tok, 0.0):.3f}')

print()
# Score the exact-code probe against its gold (doc 8) and the chatty distractor (doc 10)
code_probe = probes[0]
print(f'query: {code_probe.query}')
print(f'BM25 score doc[8]  (exact-code gold): {bm25.score(code_probe.query, 8):.3f}')
print(f'BM25 score doc[10] (chatty distractor): {bm25.score(code_probe.query, 10):.3f}')

IDF('e') = 2.079
IDF('helios') = 0.470
IDF('the') = 0.780
IDF('telemetry') = 1.569

query: What telemetry error did Helios-7 report?
BM25 score doc[8]  (exact-code gold): 5.003
BM25 score doc[10] (chatty distractor): 2.323


**Saturation, made visible.** Repeating a query term in a document keeps adding score, but with **diminishing returns** toward a ceiling of `k1 + 1`. We compute the saturated-tf factor `tf*(k1+1)/(tf + k1)` (length-norm = 1) for growing `tf` and watch it flatten.

In [4]:
k1 = BM25_K1
print(f'saturated-tf factor tf*(k1+1)/(tf+k1) at k1={k1} (ceiling = k1+1 = {k1+1}):')
for tf in [1, 2, 3, 5, 10, 50]:
    factor = tf * (k1 + 1) / (tf + k1)
    print(f'  tf={tf:>3}: {factor:.3f}')
# the gain from tf=1->2 dwarfs tf=10->50: saturation in action
gain_early = (2*(k1+1)/(2+k1)) - (1*(k1+1)/(1+k1))
gain_late  = (50*(k1+1)/(50+k1)) - (10*(k1+1)/(10+k1))
print(f'gain tf 1->2: {gain_early:.3f}   gain tf 10->50: {gain_late:.3f}')
assert gain_early > gain_late, 'early occurrences must add more than late ones (saturation)'

saturated-tf factor tf*(k1+1)/(tf+k1) at k1=1.2 (ceiling = k1+1 = 2.2):
  tf=  1: 1.000
  tf=  2: 1.375
  tf=  3: 1.571
  tf=  5: 1.774
  tf= 10: 1.964
  tf= 50: 2.148
gain tf 1->2: 0.375   gain tf 10->50: 0.184


## Step 4 — the dense lens and the paraphrase that BM25 cannot see

The dense bi-encoder matches meaning. On the **paraphrase probe**, the gold `doc[9]` shares no content word with the query, so **BM25 scores it 0** (a miss), while the dense lens scores it well and ranks it #1. This is the lexical blind spot, isolated.

In [5]:
para_probe = probes[1]
print(f'paraphrase query: {para_probe.query}')
print(f'gold passage     : {corpus[para_probe.gold]}')

# shared content tokens between query and gold (drives BM25)
from hybrid_search import tokenize
shared = set(tokenize(para_probe.query)) & set(tokenize(corpus[para_probe.gold]))
print('shared tokens (query ∩ gold):', sorted(shared) or 'NONE')

bm25_gold = bm25.score(para_probe.query, para_probe.gold)
dense_gold = float(dense.all_scores(para_probe.query)[para_probe.gold])
print(f'BM25 score of gold : {bm25_gold:.3f}   (0 -> BM25 cannot retrieve it)')
print(f'dense score of gold: {dense_gold:.3f}')
assert bm25_gold == 0.0, 'a zero-overlap paraphrase must score exactly 0 under BM25'

paraphrase query: How did the vehicle gain height after blast-off?
gold passage     : Climbing steadily, Helios-7 rose skyward moments past liftoff.
shared tokens (query ∩ gold): NONE
BM25 score of gold : 0.000   (0 -> BM25 cannot retrieve it)
dense score of gold: 0.325


## Step 5 — where each lens fails, side by side

Now run **both** lenses on **both** probes and print the rank each gives the gold passage. The pattern: dense ranks the exact-code gold **#2** (a distractor wins); BM25 **misses** the paraphrase gold (rank far below top-k). Each lens fails a different query type.

In [6]:
def gold_rank(indices, gold):
    return f'#{list(indices).index(gold)+1}' if gold in indices else 'MISS'

for p in probes:
    d_res = dense.search(p.query, k=TOP_K)
    s_res = bm25.search(p.query, k=TOP_K)
    print(f'[{p.label}]  gold=doc[{p.gold}]')
    print(f'   DENSE  top-{TOP_K}: {list(d_res.indices)}   gold rank {gold_rank(d_res.indices, p.gold)}')
    print(f'   SPARSE top-{TOP_K}: {list(s_res.indices)}   gold rank {gold_rank(s_res.indices, p.gold)}')
    print()

[exact-code (dense ranks a distractor first)]  gold=doc[8]
   DENSE  top-3: [10, 8, 0]   gold rank #2
   SPARSE top-3: [8, 10, 9]   gold rank #1

[paraphrase (BM25 scores it 0)]  gold=doc[9]
   DENSE  top-3: [9, 1, 0]   gold rank #1
   SPARSE top-3: [5, 0, 10]   gold rank MISS



![Each lens misses one query type; hybrid catches both. Bars show the rank the correct passage receives under dense, sparse (BM25), and the two fusions, for both probes (lower is better; rank 1 at top). Dense ranks the exact-code gold #2; BM25 misses the paraphrase gold entirely; hybrid recovers both. Generated by `make_figures_05.py`.](../../images/rag05_lens_miss_catch.png)

## Step 6 — the scale-mismatch trap: why you can't just ADD the raw scores

Cosine lives in `[-1, 1]`; BM25 is unbounded (here up to ~5). A **naive sum** is decided almost entirely by BM25's larger magnitude, so the dense signal is ignored. We measure BM25's share of the naive-sum magnitude, and show a naive add **collapsing** the dense #1 to a much worse rank. The fix: **min-max normalize each lens to [0,1] first**.

In [7]:
probe = probes[1]  # paraphrase probe: dense ranks the gold #1
ds = dense.all_scores(probe.query)
ss = bm25.all_scores(probe.query)
print(f'raw cosine range : [{ds.min():+.3f}, {ds.max():+.3f}]   (bounded, subset of [-1, 1])')
print(f'raw BM25 range   : [{ss.min():+.3f}, {ss.max():+.3f}]   (unbounded, >= 0)')
bm25_share = np.abs(ss).mean() / (np.abs(ds).mean() + np.abs(ss).mean()) * 100
print(f'in a NAIVE sum, BM25 supplies {bm25_share:.0f}% of the magnitude -> it drowns out cosine')

def rank_of(scores, doc):
    return list(map(int, np.argsort(scores)[::-1])).index(doc) + 1

print(f'gold dense rank      : #{rank_of(ds, probe.gold)}')
print(f'gold NAIVE-add rank  : #{rank_of(ds + ss, probe.gold)}   (collapsed by BM25 magnitude)')
print(f'gold NORMALIZED rank : #{rank_of(min_max_normalize(ds) + min_max_normalize(ss), probe.gold)}')
assert rank_of(ds, probe.gold) == 1 and rank_of(ds + ss, probe.gold) != 1, 'naive add must demote the dense #1'

raw cosine range : [-0.039, +0.325]   (bounded, subset of [-1, 1])
raw BM25 range   : [+0.000, +1.039]   (unbounded, >= 0)
in a NAIVE sum, BM25 supplies 87% of the magnitude -> it drowns out cosine
gold dense rank      : #1
gold NAIVE-add rank  : #6   (collapsed by BM25 magnitude)
gold NORMALIZED rank : #6


![Incomparable scales: each dot is one passage's score on one query. Cosine (top row) clusters in [0, 0.83]; BM25 (bottom) spans [0, 5]. The BM25 magnitudes are several times larger, so a naive sum is decided by BM25 alone — normalize before summing. Generated by `make_figures_05.py`.](../../images/rag05_scale_mismatch.png)

## Step 7 — fusion #1: weighted sum (the `alpha` dial), and tuning it

Weighted-sum fusion blends the two **normalized** scores: `alpha * dense + (1 - alpha) * sparse`. `alpha=1` is pure dense, `alpha=0` pure sparse, `alpha=0.5` equal (Weaviate's `alpha`). We sweep `alpha` and watch the optimum land **inside** the interval — `alpha=0.5` is a starting guess, not the answer.

In [8]:
sweep = alpha_sweep(probes, dense, bm25, alphas=(0.0, 0.3, 0.5, 0.6, 0.7, 1.0))
print(f"{'alpha':>6} | {'MRR':>6} | {'recall@'+str(TOP_K):>9} | note")
print('-' * 44)
for a, (mrr, rec) in sweep.items():
    note = '= pure sparse' if a == 0.0 else ('= pure dense' if a == 1.0 else ('<- best blend' if (mrr, rec) == max(sweep.values()) else ''))
    print(f'{a:>6.1f} | {mrr:>6.3f} | {rec:>9.3f} | {note}')

best = max(sweep, key=lambda a: sweep[a])
print(f'\nbest alpha = {best} (interior optimum -- neither pure lens)')
assert 0.0 < best < 1.0, 'the best alpha must be interior -- the whole point of fusing'

 alpha |    MRR |  recall@3 | note
--------------------------------------------
   0.0 |  0.500 |     0.500 | = pure sparse
   0.3 |  0.500 |     0.500 | 
   0.5 |  0.500 |     0.500 | 
   0.6 |  1.000 |     1.000 | <- best blend
   0.7 |  1.000 |     1.000 | <- best blend
   1.0 |  0.750 |     1.000 | = pure dense

best alpha = 0.6 (interior optimum -- neither pure lens)


![Tuning alpha: MRR and recall@3 over the probe set as alpha sweeps 0 (pure BM25) to 1 (pure dense). The best score sits at an interior alpha (~0.6–0.7), not at either endpoint and not at 0.5 — which is why alpha is tuned per corpus. Generated by `make_figures_05.py`.](../../images/rag05_alpha_sweep.png)

**The intuition GIF — the dial in motion.** As `alpha` sweeps, the two fused rankings re-order live; only the blended middle (`alpha ≈ 0.6`) puts **both** golds at #1, which neither pure lens manages.

![Animated — the alpha dial sweeps from pure sparse to pure dense. Left panel: the exact-code probe; right: the paraphrase probe. At alpha=0 the paraphrase gold is stranded at #6; at alpha=1 the exact-code gold drops to #2; only the blend (alpha≈0.6) puts both golds at #1. Generated by `make_animation_05.py`.](../../images/rag05_alpha_fusion.gif)

## Step 8 — fusion #2: Reciprocal Rank Fusion (RRF)

RRF ignores raw scores and uses only each list's **ranking**: a doc's fused score is `sum over lists of 1/(k + rank)`. Because it's rank-based, it needs **no normalization** and is immune to the scale mismatch — its key advantage. The constant `k` (default **60**) damps low ranks. We fuse the two lenses and confirm RRF never catastrophically misses (recall@3 = 1.0).

In [9]:
for p in probes:
    ds = dense.all_scores(p.query)
    ss = bm25.all_scores(p.query)
    rrf = reciprocal_rank_fusion([ds, ss], k_rrf=RRF_K, k=TOP_K)
    print(f'[{p.label}] RRF top-{TOP_K}: {list(rrf.indices)}   gold rank {gold_rank(rrf.indices, p.gold)}')

# The k=60 weight curve: rank 1 is worth far more than rank 10
print(f'\nRRF contribution 1/(k+rank) at k={RRF_K}:')
for r in [1, 2, 5, 10, 100]:
    print(f'  rank {r:>3}: {1/(RRF_K+r):.5f}')

[exact-code (dense ranks a distractor first)] RRF top-3: [10, 8, 0]   gold rank #2
[paraphrase (BM25 scores it 0)] RRF top-3: [0, 9, 10]   gold rank #2

RRF contribution 1/(k+rank) at k=60:
  rank   1: 0.01639
  rank   2: 0.01613
  rank   5: 0.01538
  rank  10: 0.01429
  rank 100: 0.00625


![RRF mechanics. Left: the weight 1/(k+rank) by rank for k=10/60/200 — being #1 is worth far more than #10, and a larger k flattens the curve. Right: a worked fusion of two ranked lists — a doc ranked #2/#1 overtakes one ranked #1/#2, so "strong in both" beats "strong in one." Generated by `make_figures_05.py`.](../../images/rag05_rrf_heatmap.png)

## Step 9 — the payoff: aggregate MRR and recall@k

Put it together. Over **both** probes: BM25 alone has **recall@3 = 0.5** (misses paraphrase); dense alone has **MRR = 0.75** (loses the exact-code probe). The **tuned weighted-sum hybrid (alpha=0.6)** reaches **MRR = 1.0 and recall@3 = 1.0** — strictly better than either single lens. RRF reaches **recall@3 = 1.0** (robust, scale-free, no tuning).

In [10]:
results = evaluate(probes, dense, bm25, alpha=TUNED_ALPHA)
print(f"{'method':<30} | {'MRR':>6} | {'recall@'+str(TOP_K):>9}")
print('-' * 54)
for name, (mrr, rec) in results.items():
    print(f'{name:<30} | {mrr:>6.3f} | {rec:>9.3f}')

dense_mrr, dense_rec = results['dense only']
sparse_mrr, sparse_rec = results['sparse only (BM25)']
ws_mrr, ws_rec = results[f'hybrid weighted (a={TUNED_ALPHA})']
rrf_mrr, rrf_rec = results[f'hybrid RRF (k={RRF_K})']

# Correctness BEFORE the claim
assert sparse_rec < 1.0, 'BM25 alone misses the paraphrase probe'
assert dense_mrr < 1.0, 'dense alone ranks the exact-code gold #2'
assert ws_rec == 1.0 and ws_mrr > dense_mrr and ws_mrr > sparse_mrr, 'tuned hybrid must beat both lenses'
assert rrf_rec == 1.0, 'RRF must catch both probes'
print('\nhybrid beats BOTH single lenses (weighted MRR & recall strictly higher): True')

method                         |    MRR |  recall@3
------------------------------------------------------
dense only                     |  0.750 |     1.000
sparse only (BM25)             |  0.500 |     0.500
hybrid weighted (a=0.6)        |  1.000 |     1.000
hybrid RRF (k=60)              |  0.500 |     1.000

hybrid beats BOTH single lenses (weighted MRR & recall strictly higher): True


## Step 10 — the library one-liners

In production you don't hand-roll fusion. Each engine exposes the same two families:

```python
# Weaviate — hybrid with an alpha dial (0 = pure BM25, 1 = pure vector, 0.5 = equal)
collection.query.hybrid(query="telemetry error", alpha=0.6, limit=3)

# Elasticsearch / OpenSearch — RRF over a BM25 retriever and a kNN retriever
# { "retriever": { "rrf": { "retrievers": [ {"standard": {...BM25...}},
#                                           {"knn": {...dense...}} ],
#                           "rank_constant": 60, "rank_window_size": 100 } } }

# Qdrant — query API with prefetch + fusion
# client.query_points(prefetch=[dense_query, sparse_query], query=Fusion.RRF)

# pgvector + Postgres full-text — combine ts_rank (BM25-like) and 1 - (embedding <=> q)
# Pinecone — sparse-dense vectors in one index (dot product over both)
```

The one-liner hides exactly the mechanics we built by hand — which is why building it once is the lesson the API can't teach.

**Recap.** Two lenses, two blind spots: dense blurs exact tokens, BM25 is blind to paraphrase. Fuse them — **weighted-sum** (normalize first, then tune `alpha`) or **RRF** (rank-based, scale-free, `k=60`) — and a passage strong on *either* lens surfaces. RRF is the robust default; weighted-sum, tuned, can edge it out.